In [4]:
!pip install langchain-google-genai langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Loading API key securely from Colab secrets
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets.")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    print("Attempted to load API key from .env file.")

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("Missing GOOGLE_API_KEY — add it to Colab secrets or your .env file.")

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

print("Gemini model initialized successfully.")

API key loaded from Colab secrets.
Gemini model initialized successfully.


In [6]:
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, AnyMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

# Loading the real CSV files from the repo (same as starter code)
product_orders_df = pd.read_csv("data/Laptop Orders.csv")
product_pricing_df = pd.read_csv("data/Laptop pricing.csv")

print("Orders loaded:")
print(product_orders_df)
print("\nPricing loaded:")
print(product_pricing_df)



ORDER_STATUS_PROMPT = """
You are Alex, an Order Status specialist at ShopEasy customer support.
You help customers check their laptop order status and delivery dates.
Available orders:
{orders}
If asked about an order, look it up and provide product name, quantity ordered, and delivery date.
If the order is not found, politely say so.
Always be concise, friendly, and reassuring.
""".format(orders=product_orders_df.to_string(index=False))

REFUND_POLICY_PROMPT = """
You are Jordan, a Refund Policy specialist at ShopEasy customer support.
You explain refund and return policies clearly. Key policy:
- Returns accepted within 30 days of purchase
- Items must be unused and in original packaging
- Refunds processed in 5-7 business days
- Digital products are non-refundable
- Customer needs their order number to start a return
Be empathetic and helpful.
"""

PRODUCT_SUGGESTION_PROMPT = """
You are Sam, a Product Recommendation specialist at ShopEasy.
You help customers find the right laptop. Here are the available products with pricing:
{products}

Full product descriptions:
- AlphaBook Pro: Intel i7 12th Gen, 16GB DDR4 RAM, 1TB SSD. Best for professionals on the go.
- GammaAir X: AMD Ryzen 7, 32GB DDR4 RAM, 512GB NVMe SSD. Thin and light, high performance.
- SpectraBook S: Intel Core i9, 64GB RAM, 2TB SSD. Workstation-class for video editing and 3D rendering.
- OmegaPro G17: Ryzen 9 5900HX, 32GB RAM, 1TB SSD, 17-inch high refresh rate display. Gaming powerhouse.
- NanoEdge Flex: Apple M1 Pro, 16GB unified memory, 512GB SSD, 2-in-1 convertible. For creative professionals.

Ask about the customer's budget and use case, then suggest 1-2 products with brief reasons.
""".format(products=product_pricing_df.to_string(index=False))

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

def invoke_agent(state: AgentState, system_prompt: str) -> dict:
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def order_status_agent(state: AgentState) -> dict:
    """Agent 1: Handles order tracking inquiries."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=ORDER_STATUS_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def refund_policy_agent(state: AgentState) -> dict:
    """Agent 2: Handles refund and return policy questions."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=REFUND_POLICY_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def product_suggestion_agent(state: AgentState) -> dict:
    """Agent 3: Handles product recommendation requests."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=PRODUCT_SUGGESTION_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

print("Three customer service agents defined.")

Orders loaded:
   Order ID Product Ordered  Quantity Ordered Delivery Date
0  ORD-8276   SpectraBook S                 3    2024-10-16
1  ORD-6948    OmegaPro G17                 3    2024-10-25
2  ORD-7311   NanoEdge Flex                 2    2024-10-19
3  ORD-4633    OmegaPro G17                 2    2024-10-15
4  ORD-2050      GammaAir X                 2    2024-10-26

Pricing loaded:
            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2
Three customer service agents defined.


In [7]:
import uuid

class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        self.system_prompt = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model = model
        self.debug = debug

        router_graph = StateGraph(AgentState)

        router_graph.add_node("Router", self.call_llm)
        router_graph.add_node("Order_Agent", order_status_agent)
        router_graph.add_node("Refund_Agent", refund_policy_agent)
        router_graph.add_node("Product_Agent", product_suggestion_agent)
        router_graph.add_node("Small_Talk", self.respond_smalltalk)

        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {
                "ORDER":     "Order_Agent",
                "REFUND":    "Refund_Agent",
                "PRODUCT":   "Product_Agent",
                "SMALLTALK": "Small_Talk",
                "END": END
            }
        )

        router_graph.add_edge("Order_Agent", END)
        router_graph.add_edge("Refund_Agent", END)
        router_graph.add_edge("Product_Agent", END)
        router_graph.add_edge("Small_Talk", END)

        router_graph.set_entry_point("Router")

        memory = MemorySaver()
        self.router_graph = router_graph.compile(checkpointer=memory)

    def call_llm(self, state: AgentState) -> dict:
        """Router node: classifies the user intent."""
        messages = state["messages"]
        if self.debug:
            print(f"Router received: {messages[-1].content}")
        messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"Router decision: {result.content}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: AgentState) -> dict:
        """Small talk node: handles greetings and general chat."""
        messages = [SystemMessage(content=self.smalltalk_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        return {"messages": [AIMessage(content=result.content)]}

    def find_route(self, state: AgentState) -> str:
      """Reads the router output and returns the destination node."""
      last_message = state["messages"][-1]
      content = last_message.content
      if isinstance(content, list):
          for item in content:
              if isinstance(item, dict) and "text" in item:
                  content = item["text"]
                  break
              elif isinstance(item, str):
                  content = item
                  break
      content = str(content).strip().upper()
      first_word = content.split()[0] if content.split() else "END"

      valid = {"ORDER", "REFUND", "PRODUCT", "SMALLTALK", "END"}
      destination = first_word if first_word in valid else "END"

      if self.debug:
          print(f"Routing to: {destination}")
      return destination

print("RouterAgent class defined.")

RouterAgent class defined.


In [8]:
system_prompt = """
You are a Router that analyzes the customer's message and picks exactly ONE of these options:

ORDER     - If the query is about an order: order status, delivery date, order details.
REFUND    - If the query is about returns, refunds, or cancellations.
PRODUCT   - If the query is about product features, recommendations, or pricing.
SMALLTALK - If the message is small talk: greetings, goodbyes, general chat.
END       - Default for anything else.

Your output must be ONLY one word: ORDER, REFUND, PRODUCT, SMALLTALK, or END.
Do not add any explanation.
"""

smalltalk_prompt = """
You are a friendly customer service assistant.
Respond professionally to greetings, goodbyes, and general small talk.
Mention that you can help with: order status, refunds/returns, and product recommendations.
"""

router_agent = RouterAgent(model, system_prompt, smalltalk_prompt, debug=False)
print("Customer Service Chatbot initialized!")

Customer Service Chatbot initialized!


In [10]:
import uuid
import time

user_inputs = [
    "Hi there!",
    "What is the status of order ORD-7311?",
    "When will it be delivered?",
    "How do I return something if I don't like it?",
    "What's the return window?",
    "I'm looking for a good laptop for video editing",
    "What is the price of the SpectraBook S?",
    "Thanks, goodbye!",
]

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("=" * 60)
print("Customer Service Chatbot — Demo")
print("=" * 60)

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    #Format the user message
    user_message = {"messages":[HumanMessage(input)]}
    #Get response from the agent
    ai_response = router_agent.router_graph.invoke(user_message,config=config)
    #Print the response
    print(f"\nAGENT : {ai_response['messages'][-1].content}")
    time.sleep(13)

Customer Service Chatbot — Demo
----------------------------------------
USER : Hi there!

AGENT : Hi there! Thanks for reaching out.

I'm here to help you with a few things today. I can assist with **order status**, **refunds/returns**, or even **product recommendations**.

What can I help you with?
----------------------------------------
USER : What is the status of order ORD-7311?

AGENT : I can certainly help you with that!

Your order ORD-7311 for the **NanoEdge Flex** (quantity 2) is scheduled for delivery on **October 19, 2024**.
----------------------------------------
USER : When will it be delivered?

AGENT : ORDERYour order ORD-7311 is scheduled for delivery on **October 19, 2024**.
----------------------------------------
USER : How do I return something if I don't like it?

AGENT : I understand you're asking about our return process, and I'm happy to explain it for you! We want to make sure you're completely satisfied with your purchases from ShopEasy.

Here's a quick ove

In [3]:
import os
print(os.listdir("data"))


['Laptop pricing.csv', 'Laptop Orders.csv']


In [1]:
import os
os.makedirs("data", exist_ok=True)
print("✅ data folder created")

✅ data folder created
